In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica global + Classificação de falhas via Random Forest
Autor: Luiz Eduardo Abdala José
"""

import re, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ===================== PARÂMETROS =====================
ARQ_BASE = "base_geral.pkl"
REF_TEMP = 20
FREQ_MIN_KHZ = 30
FREQ_MAX_KHZ = 50
SMOOTH_WIN = 5
TAU_MAX_FRAC = 0.025
ANCHOR_TO_REF_ENDS = True

CAPS = dict(gain_frac=0.60, offset_frac=0.60, tilt_frac=0.40)

# ===================== FUNÇÕES AUXILIARES =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f / 1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def moving_average(arr, win):
    if win <= 1 or win % 2 == 0: return arr
    r = win // 2
    padl = np.repeat(arr[:1], r)
    padr = np.repeat(arr[-1:], r)
    x = np.concatenate([padl, arr, padr])
    c = np.cumsum(x, dtype=float)
    c = np.concatenate([[0.0], c])
    s = c[win:] - c[:-win]
    return s / float(win)

def shift_interp(x_row, fhz, tau_hz):
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

def energy_weighted_centroid(f, x):
    xm = np.asarray(x, float)
    w = xm * xm
    den = float(np.trapezoid(w, f))
    if den <= 1e-18: return float(np.mean(f))
    num = float(np.trapezoid(f * w, f))
    return num / den

def slope_over_band(f, x):
    return float((x[-1] - x[0]) / (f[-1] - f[0] + 1e-12))

def compute_features(X, f):
    X = np.asarray(X, float); n, m = X.shape
    out = []
    for i in range(n):
        x = X[i]
        mean = float(np.mean(x))
        std = float(np.std(x))
        amp = float(x.max() - x.min())
        slope = slope_over_band(f, x)
        centroid = energy_weighted_centroid(f, x)
        out.append([mean, std, amp, slope, centroid])
    cols = ["mean", "std", "amp", "slope", "centroid"]
    return np.array(out, float), cols

def fit_feature_vs_temp_models(F, T):
    models = {}
    T = np.asarray(T, float).reshape(-1, 1)
    for j, name in enumerate(["mean", "std", "amp", "slope", "centroid"]):
        rf = RandomForestRegressor(n_estimators=100, random_state=0)
        rf.fit(T, F[:, j])
        models[name] = rf
    return models

def feature_targets_at_ref(models, ref_temp=REF_TEMP):
    Tref = np.array([[ref_temp]])
    return {name: float(m.predict(Tref)[0]) for name, m in models.items()}

def apply_compensation_by_features(x, f, targets, caps, y_ref=None):
    x = x.copy()
    mean_t = targets["mean"]; amp_t = targets["amp"]; slope_t = targets["slope"]
    centroid_t = targets["centroid"]
    mean_x = float(x.mean()); amp_x = float(x.max() - x.min()); slope_x = slope_over_band(f, x)
    offset = mean_t - mean_x
    offset_cap = caps["offset_frac"] * max(1e-9, amp_x)
    offset = float(np.clip(offset, -offset_cap, offset_cap))
    x = x + offset
    gain = 1.0 if amp_x <= 1e-9 else float(amp_t / amp_x)
    gmin = 1.0 - caps["gain_frac"]; gmax = 1.0 + caps["gain_frac"]
    gain = float(np.clip(gain, gmin, gmax))
    x = mean_t + gain * (x - mean_t)
    delta_slope = slope_t - slope_x
    u = np.linspace(-0.5, 0.5, len(x))
    df = (f[-1] - f[0] + 1e-12)
    tilt_signal = (delta_slope * df) * u
    tilt_cap = caps["tilt_frac"] * max(1e-9, amp_x)
    tilt_signal = np.clip(tilt_signal, -tilt_cap, tilt_cap)
    x = x + tilt_signal
    cent_x = energy_weighted_centroid(f, x)
    delta_c = centroid_t - cent_x
    tau_max = TAU_MAX_FRAC * (f[-1] - f[0])
    tau = float(np.clip(delta_c, -tau_max, tau_max))
    if abs(tau) > 1e-12:
        x = shift_interp(x, f, tau)
    if ANCHOR_TO_REF_ENDS and (y_ref is not None):
        e0 = x[0] - y_ref[0]; e1 = x[-1] - y_ref[-1]
        corr = np.linspace(e0, e1, len(x))
        x = x - corr
    return x

def compensate_all(df, fcols, fhz, ref_temp=REF_TEMP):
    print("🔹 Calculando modelo de features vs temperatura...")
    X = df[fcols].to_numpy(float)
    T = df["temperatura_c"].to_numpy(float)
    F, _ = compute_features(X, fhz)
    feat_models = fit_feature_vs_temp_models(F, T)
    targets = feature_targets_at_ref(feat_models, ref_temp)
    pool_ref = df.loc[np.isclose(df["temperatura_c"], ref_temp), fcols].to_numpy(float)
    y_ref = np.median(pool_ref, axis=0) if len(pool_ref) > 0 else X.mean(axis=0)
    print("🔹 Aplicando compensação para toda a base...")
    Y = np.zeros_like(X)
    for i in range(X.shape[0]):
        y = apply_compensation_by_features(X[i], fhz, targets, CAPS, y_ref=y_ref)
        if SMOOTH_WIN > 1 and SMOOTH_WIN % 2 == 1:
            y = moving_average(y, SMOOTH_WIN)
        Y[i] = y
    df_comp = df.copy()
    df_comp[fcols] = Y
    print("✅ Compensação concluída.")
    return df_comp, y_ref

# ===================== EXECUÇÃO PRINCIPAL =====================
print("🔹 Carregando base de dados completa...")
df = pd.read_pickle(ARQ_BASE)

# Separa bandas válidas de frequência
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
print(f"Faixa selecionada: {len(fcols)} colunas de frequência ({FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz)")

# Aplica compensação
df_comp, y_ref = compensate_all(df, fcols, fhz, REF_TEMP)
df_comp.to_pickle("base_geral_compensada.pkl")
print("✅ Base compensada salva em: pklresultados/base_geral_compensada.pkl")

# ===================== CLASSIFICAÇÃO DE FALHAS =====================
print("\n🔹 Treinando Random Forest para detecção de falhas...")
X = df_comp[fcols].to_numpy(float)
y = df_comp["falha"].to_numpy(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train_s, y_train)
y_pred = clf.predict(X_test_s)

print("\n== RESULTADOS RANDOM FOREST (após compensação) ==")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

acc = (y_pred == y_test).mean()
print(f"Acurácia total: {acc*100:.2f}%")

# ===================== VISUALIZAÇÃO DE EXEMPLO =====================
plt.figure(figsize=(9,5))
plt.plot(fhz/1e3, y_ref, '--', c='black', lw=1.2, label=f"Ref {REF_TEMP}°C")
plt.plot(fhz/1e3, X_test[0], c='tab:red', alpha=0.6, label="Original (amostra teste)")
plt.plot(fhz/1e3, scaler.inverse_transform(X_test_s)[0], c='tab:blue', lw=2, label="Normalizado")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real (normalizada)")
plt.legend(); plt.tight_layout(); plt.show()
